# 🛡️ Sentinel — Category Classifier v2

**Constraint:** `bge-base-en-v1.5` stays frozen (same model as the FAISS index — changing it would require rebuilding the entire index).

**Improvement over v1:** replace sklearn's basic `MLPClassifier` with a proper PyTorch MLP — dropout, BatchNorm, residual connections, AdamW + cosine LR schedule, early stopping, label smoothing.

|                   | v1                        | v2                                       |
| ----------------- | ------------------------- | ---------------------------------------- |
| Encoder           | Frozen `bge-base-en-v1.5` | Frozen `bge-base-en-v1.5` (unchanged)    |
| Classifier        | sklearn MLPClassifier     | PyTorch MLP + dropout + BN + residuals   |
| Training          | `clf.fit()`               | AdamW + cosine schedule + early stopping |
| FAISS index       | ✅ Compatible             | ✅ Compatible (nothing changed)          |
| Expected accuracy | ~71%                      | ~76–80%                                  |

**Workflow:**

1. Upload your `train.jsonl`, `val.jsonl`, `test.jsonl` in cell 4
2. Auxiliary data (chunked DB, etc.) pulled from HuggingFace in cell 5
3. Run all cells — artifact saved at the end


## 0. Runtime check


In [ ]:
import subprocess, torch
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print(result.stdout.strip())
print(f'CUDA: {torch.cuda.is_available()} | BF16: {torch.cuda.is_bf16_supported()} | PyTorch: {torch.__version__}')

## 1. Install dependencies


In [ ]:
!pip install -q \
    sentence-transformers \
    huggingface_hub \
    datasets \
    scikit-learn \
    joblib \
    seaborn \
    matplotlib \
    tqdm

## 2. Configuration


In [ ]:
# ─── HuggingFace (for auxiliary data only) ────────────────────────────────────
HF_DATASET_NAME   = "EXANU/antiplagiator-artifacts"

# ─── Model (must match the FAISS index) ──────────────────────────────────────
EMBED_MODEL       = "BAAI/bge-base-en-v1.5"
EMBED_DIM         = 768
MAX_LENGTH        = 512
ENCODE_BATCH_SIZE = 256   # H100 can handle large encode batches easily

# ─── Dataset column names ────────────────────────────────────────────────────
TEXT_COL     = "title"
ABSTRACT_COL = "abstract"      # set None if your JSONL has no abstract
LABEL_COL    = "top_category_name"
MIN_SAMPLES  = 30

# ─── PyTorch MLP ─────────────────────────────────────────────────────────────
HIDDEN_DIMS  = [1024, 512, 256]   # deeper than v1's (512,256,128)
DROPOUT      = 0.35
LR           = 3e-4
WEIGHT_DECAY = 1e-4
EPOCHS       = 60
BATCH_SIZE   = 2048             # pure MLP training — huge batches are fine
PATIENCE     = 8                # early stopping patience (epochs)
LABEL_SMOOTH = 0.05             # softens overconfident predictions
SEED         = 42

# ─── Output ──────────────────────────────────────────────────────────────────
OUTPUT_DIR        = "/content/sentinel-v2"
ARTIFACT_PATH     = "/content/category_classifier_v2.pkl"
# ──────────────────────────────────────────────────────────────────────────────

import os, random, json
import numpy as np
import torch
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 3. HuggingFace login (for auxiliary data)


In [ ]:
from huggingface_hub import login
login(token=HF_TOKEN)

## 4. Upload your train / val / test files

Run this cell — a file picker will appear. Select your `train.jsonl`, `val.jsonl`, and `test.jsonl` files.


In [ ]:
from google.colab import files as colab_files

print("Upload train.jsonl, val.jsonl, test.jsonl (select all 3 at once):")
uploaded = colab_files.upload()

# Save uploaded files and confirm
for fname, data in uploaded.items():
    path = f"/content/{fname}"
    with open(path, "wb") as f:
        f.write(data)
    lines = open(path).readlines()
    print(f"  ✓ {fname}: {len(lines):,} records")

# Resolve paths (handles any filename the user uploaded)
def find_split(keyword):
    for name in uploaded:
        if keyword in name.lower():
            return f"/content/{name}"
    raise FileNotFoundError(f"Could not find a file matching '{keyword}'. Check your filenames.")

TRAIN_PATH = find_split("train")
VAL_PATH   = find_split("val")
TEST_PATH  = find_split("test")
print(f"\nTrain: {TRAIN_PATH}")
print(f"Val:   {VAL_PATH}")
print(f"Test:  {TEST_PATH}")

## 5. Load auxiliary data from HuggingFace

This is for any extra data stored on your HF Hub (chunked DB, pre-built indexes, etc.).  
**Skip this cell if you don't need auxiliary data for the classifier.**


In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download

# Example: download a specific file from your HF dataset repo
# chunked_db_path = hf_hub_download(
#     repo_id=HF_DATASET_NAME,
#     filename="chunked_database.jsonl",
#     repo_type="dataset",
#     token=HF_TOKEN,
#     local_dir="/content/hf_data"
# )
# print(f"Chunked DB: {chunked_db_path}")

# Or download the entire repo:
# snapshot_download(repo_id=HF_DATASET_NAME, repo_type="dataset", token=HF_TOKEN, local_dir="/content/hf_data")

print("Auxiliary data cell — uncomment and configure as needed.")

## 6. Parse JSONL splits


In [ ]:
from collections import Counter

def load_jsonl(path):
    texts, labels = [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            d = json.loads(line)
            title    = str(d.get(TEXT_COL, "")).strip()
            abstract = str(d.get(ABSTRACT_COL, "")).strip() if ABSTRACT_COL else ""
            label    = str(d.get(LABEL_COL, "")).strip()
            if not label:
                continue
            text = f"{title}. {abstract}".strip(". ") if abstract else title
            texts.append(text)
            labels.append(label)
    return texts, labels

train_texts, train_labels = load_jsonl(TRAIN_PATH)
val_texts,   val_labels   = load_jsonl(VAL_PATH)
test_texts,  test_labels  = load_jsonl(TEST_PATH)

print(f"Train: {len(train_texts):,} | Val: {len(val_texts):,} | Test: {len(test_texts):,}")

# Drop sparse classes
dist = Counter(train_labels)
valid_classes = {c for c, n in dist.items() if n >= MIN_SAMPLES}
dropped = {c: n for c, n in dist.items() if c not in valid_classes}
if dropped:
    print(f"Dropping {len(dropped)} sparse classes: {dropped}")

def filter_split(texts, labels):
    pairs = [(t, l) for t, l in zip(texts, labels) if l in valid_classes]
    return zip(*pairs) if pairs else ([], [])

train_texts, train_labels = [list(x) for x in filter_split(train_texts, train_labels)]
val_texts,   val_labels   = [list(x) for x in filter_split(val_texts,   val_labels)]
test_texts,  test_labels  = [list(x) for x in filter_split(test_texts,  test_labels)]

print(f"After filter — Train: {len(train_texts):,} | Val: {len(val_texts):,} | Test: {len(test_texts):,}")
print(f"Classes: {len(valid_classes)}")

## 7. Encode with bge-base-en-v1.5 (frozen)


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import LabelEncoder

print(f"Loading {EMBED_MODEL} ...")
embed_model = SentenceTransformer(EMBED_MODEL, device=str(DEVICE))
embed_model.eval()
for p in embed_model.parameters():
    p.requires_grad_(False)  # encoder fully frozen

def encode(texts, desc=""):
    return embed_model.encode(
        texts,
        batch_size=ENCODE_BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

print("Encoding train ...")
X_train = encode(train_texts)
print("Encoding val ...")
X_val   = encode(val_texts)
print("Encoding test ...")
X_test  = encode(test_texts)

print(f"\nEmbedding shapes — train: {X_train.shape} | val: {X_val.shape} | test: {X_test.shape}")

# Label encoding
le = LabelEncoder()
le.fit(sorted(valid_classes))
y_train = le.transform(train_labels)
y_val   = le.transform(val_labels)
y_test  = le.transform(test_labels)
NUM_CLASSES = len(le.classes_)
print(f"Label encoder fitted — {NUM_CLASSES} classes")

## 8. Class weights


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

cw = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=y_train)
class_weights = torch.tensor(cw, dtype=torch.float32).to(DEVICE)

print("Top 5 up-weighted classes:")
for i in np.argsort(cw)[::-1][:5]:
    print(f"  {le.classes_[i]:<50}  weight={cw[i]:.3f}")

## 9. PyTorch MLP with residuals, BatchNorm, dropout

Key improvements over sklearn's `MLPClassifier`:

- Residual connections (stabilise deep training)
- BatchNorm (faster convergence, acts as regularisation)
- Higher dropout (0.35) with a deeper network
- Label smoothing (reduces overconfidence on hard boundary classes)
- Cosine LR decay + early stopping


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
        )
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(x + self.net(x))


class SentinelMLP(nn.Module):
    def __init__(self, in_dim, hidden_dims, num_classes, dropout):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.GELU(),
                nn.Dropout(dropout),
            ]
            if prev == h:           # residual only when dims match
                layers.append(ResidualBlock(h, dropout))
            prev = h
        self.trunk = nn.Sequential(*layers)
        self.head  = nn.Linear(prev, num_classes)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.head(self.trunk(x))


mlp = SentinelMLP(EMBED_DIM, HIDDEN_DIMS, NUM_CLASSES, DROPOUT).to(DEVICE)
total = sum(p.numel() for p in mlp.parameters())
print(f"MLP parameters: {total:,}")
print(mlp)

## 10. DataLoaders


In [ ]:
from torch.utils.data import TensorDataset, DataLoader

def make_loader(X, y, shuffle=False):
    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.long)
    )
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=0, pin_memory=True)

train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader   = make_loader(X_val,   y_val)
test_loader  = make_loader(X_test,  y_test)
print(f"Batches — train: {len(train_loader)} | val: {len(val_loader)} | test: {len(test_loader)}")

## 11. Training loop


In [ ]:
from tqdm.auto import tqdm
import torch.nn.functional as F

# ── Focal Loss (replaces CrossEntropy — focuses on hard examples) ─────────────
class FocalLoss(torch.nn.Module):
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.05):
        super().__init__()
        self.weight = weight
        self.gamma  = gamma
        self.ls     = label_smoothing

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight,
                             label_smoothing=self.ls, reduction='none')
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()

loss_fn = FocalLoss(weight=class_weights, gamma=2.0, label_smoothing=LABEL_SMOOTH)

# ── Mixup augmentation (smooths decision boundaries between overlapping classes)
def mixup_batch(X, y, alpha=0.2):
    lam   = float(np.random.beta(alpha, alpha))
    idx   = torch.randperm(X.size(0), device=X.device)
    X_mix = lam * X + (1 - lam) * X[idx]
    return X_mix, y, y[idx], lam

optimizer = torch.optim.AdamW(mlp.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR * 0.01)

best_val_acc  = 0.0
best_state    = None
patience_cnt  = 0
history       = {"train_loss": [], "val_loss": [], "val_acc": []}

for epoch in range(1, EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────────────
    mlp.train()
    running_loss = 0.0
    for X_batch, y_batch in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False):
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        X_mix, y_a, y_b, lam = mixup_batch(X_batch, y_batch)
        optimizer.zero_grad()
        logits = mlp(X_mix)
        loss   = lam * loss_fn(logits, y_a) + (1 - lam) * loss_fn(logits, y_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(mlp.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item() * len(X_batch)

    train_loss = running_loss / len(X_train)

    # ── Validate ───────────────────────────────────────────────────────────
    mlp.eval()
    val_loss_sum, val_correct = 0.0, 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            logits        = mlp(X_batch)
            val_loss_sum  += loss_fn(logits, y_batch).item() * len(X_batch)
            val_correct   += (logits.argmax(1) == y_batch).sum().item()

    val_loss = val_loss_sum / len(X_val)
    val_acc  = val_correct  / len(X_val)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    scheduler.step()

    # ── Early stopping ─────────────────────────────────────────────────────
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state   = {k: v.cpu().clone() for k, v in mlp.state_dict().items()}
        patience_cnt = 0
        tag = " \u2190 best"
    else:
        patience_cnt += 1
        tag = f" (patience {patience_cnt}/{PATIENCE})"

    print(f"Epoch {epoch:3d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}{tag}")

    if patience_cnt >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}.")
        break

mlp.load_state_dict(best_state)
print(f"\nBest val accuracy: {best_val_acc:.4f}")


## 11b. Per-class threshold tuning

Finds the optimal decision threshold per class on the val set.
Fixes EESS over-prediction (threshold raised) and CS under-prediction (threshold lowered).


In [ ]:
from sklearn.metrics import f1_score as sk_f1

# Collect val softmax probabilities
mlp.eval()
val_probs_list, val_true_list = [], []
with torch.no_grad():
    for X_batch, y_batch in val_loader:
        probs = torch.softmax(mlp(X_batch.to(DEVICE)), dim=-1)
        val_probs_list.append(probs.cpu())
        val_true_list.extend(y_batch.numpy())

val_probs_all = torch.cat(val_probs_list).numpy()
val_true_all  = np.array(val_true_list)

# Grid-search best threshold per class
thresholds   = np.arange(0.05, 0.95, 0.02)
best_thresh  = np.full(NUM_CLASSES, 0.5)

print(f"{'Class':<50} {'Threshold':>10} {'Val F1':>8}")
print("-" * 72)
for c in range(NUM_CLASSES):
    best_f1, best_t = 0.0, 0.5
    for t in thresholds:
        preds_c = (val_probs_all[:, c] >= t).astype(int)
        true_c  = (val_true_all == c).astype(int)
        f = sk_f1(true_c, preds_c, zero_division=0)
        if f > best_f1:
            best_f1, best_t = f, t
    best_thresh[c] = best_t
    print(f"  {le.classes_[c]:<48} {best_t:>10.2f} {best_f1:>8.3f}")

print("\nPer-class thresholds tuned.")

# Helper used at test time and in the artifact
def predict_with_thresholds(probs_np, thresh):
    """Pick the class with the highest margin above its threshold."""
    margins = probs_np - thresh[np.newaxis, :]
    return np.argmax(margins, axis=1)


## 12. Training curves


In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history["train_loss"], label="train", color="#378ADD")
ax1.plot(history["val_loss"],   label="val",   color="#D85A30")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Loss")
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot([v * 100 for v in history["val_acc"]], color="#1D9E75")
ax2.axhline(71.0, linestyle="--", color="gray", linewidth=0.8, label="v1 baseline (71%)")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Val accuracy (%)"); ax2.set_title("Val accuracy")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/training_curves.png", dpi=150)
plt.show()

## 13. Test set evaluation


In [ ]:
from sklearn.metrics import classification_report

mlp.eval()
all_probs = []
with torch.no_grad():
    for X_batch, _ in test_loader:
        probs = torch.softmax(mlp(X_batch.to(DEVICE)), dim=-1)
        all_probs.append(probs.cpu())

test_probs = torch.cat(all_probs).numpy()

# Standard argmax predictions
preds_std = test_probs.argmax(axis=1)

# Threshold-adjusted predictions
preds = predict_with_thresholds(test_probs, best_thresh)

report_str  = classification_report(y_test, preds, target_names=le.classes_)
report_dict = classification_report(y_test, preds, target_names=le.classes_, output_dict=True)

print("=== With per-class thresholds ===")
print(report_str)

v1_acc  = 0.7096
v2_std  = preds_std.tolist().count(int(y_test[i])) / len(y_test)  # quick check
v2_acc  = report_dict["accuracy"]
print(f"v1 test accuracy:                    {v1_acc:.4f}")
print(f"v2 test accuracy (argmax):           {float((preds_std == y_test).mean()):.4f}")
print(f"v2 test accuracy (thresholded):      {v2_acc:.4f}  (+{(v2_acc - v1_acc)*100:.2f}pp vs v1)")

with open(f"{OUTPUT_DIR}/test_metrics_v2.json", "w") as f:
    json.dump(report_dict, f, indent=2)


## 14. Confusion matrix


In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

short = [
    c.replace("High Energy Physics - ", "HEP-")
     .replace("General Relativity and Quantum Cosmology", "GR&QC")
     .replace("Electrical Engineering and Systems Science", "EESS")
     .replace("Quantitative ", "Q.")
    for c in le.classes_
]

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=short, yticklabels=short, ax=ax,
            vmin=0, vmax=1, linewidths=0.3, linecolor="#eee")
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("True", fontsize=12)
ax.set_title("Confusion matrix (normalised) — test set v2", fontsize=13)
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/confusion_matrix_v2.png", dpi=150)
plt.show()

## 15. v1 vs v2 F1 comparison


In [ ]:
v1_f1 = {
    "Astrophysics": 0.853, "Computer Science": 0.757, "Condensed Matter": 0.734,
    "Economics": 0.695, "Electrical Engineering and Systems Science": 0.594,
    "General Relativity and Quantum Cosmology": 0.749, "High Energy Physics - Experiment": 0.683,
    "High Energy Physics - Lattice": 0.829, "High Energy Physics - Phenomenology": 0.790,
    "High Energy Physics - Theory": 0.695, "Mathematical Physics": 0.379,
    "Mathematics": 0.623, "Nonlinear Sciences": 0.552, "Nuclear Experiment": 0.704,
    "Nuclear Theory": 0.714, "Physics": 0.534, "Quantitative Biology": 0.630,
    "Quantitative Finance": 0.762, "Quantum Physics": 0.784, "Statistics": 0.688
}

classes_sorted = sorted(v1_f1, key=lambda c: v1_f1[c])
v1_vals = [v1_f1[c] for c in classes_sorted]
v2_vals = [report_dict.get(c, {}).get("f1-score", 0) for c in classes_sorted]

short_s = [c.replace("High Energy Physics - ", "HEP-")
             .replace("General Relativity and Quantum Cosmology", "GR&QC")
            for c in classes_sorted]

fig, ax = plt.subplots(figsize=(10, 8))
y = range(len(classes_sorted))
ax.barh(y,                      v1_vals, height=0.4, color="#B5D4F4", label="v1 (sklearn MLP)")
ax.barh([i + 0.4 for i in y],  v2_vals, height=0.4, color="#378ADD", label="v2 (PyTorch MLP)")
ax.set_yticks([i + 0.2 for i in y])
ax.set_yticklabels(short_s, fontsize=9)
ax.set_xlabel("F1-score")
ax.set_title("Per-class F1: v1 vs v2")
ax.set_xlim(0, 1)
ax.legend()
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/v1_vs_v2_f1.png", dpi=150)
plt.show()

print("\nΔ F1 per class (v2 − v1):")
for c, v1, v2 in sorted(zip(classes_sorted, v1_vals, v2_vals), key=lambda x: -(x[2]-x[1])):
    delta = v2 - v1
    bar = "█" * int(abs(delta) * 40)
    sign = "+" if delta >= 0 else "-"
    print(f"  {c:<50} {sign}{abs(delta):.3f}  {bar}")

## 16. Save drop-in artifact for the Sentinel engine

The engine loads `category_classifier.pkl` and calls `.predict(texts)`.  
We export a wrapper that replicates that exact interface — **no engine code changes needed**.


In [ ]:
import joblib, copy

class SentinelMLPWrapper:
    """
    Drop-in replacement for the sklearn MLPClassifier.
    Exposes .predict() and .predict_proba() using the trained PyTorch MLP
    on top of frozen bge-base-en-v1.5 embeddings.
    """
    def __init__(self, mlp_state, hidden_dims, embed_model_name, label_encoder, embed_dim=768, dropout=0.0):
        self.mlp_state       = mlp_state
        self.hidden_dims     = hidden_dims
        self.embed_model_name = embed_model_name
        self.le              = label_encoder
        self.embed_dim       = embed_dim
        self._mlp            = None
        self._embedder       = None
        self._device         = None

    def _load(self, device="cpu"):
        if self._mlp is None:
            from sentence_transformers import SentenceTransformer
            self._device   = torch.device(device)
            self._embedder = SentenceTransformer(self.embed_model_name, device=device)
            self._embedder.eval()
            self._mlp = SentinelMLP(self.embed_dim, self.hidden_dims, len(self.le.classes_), dropout=0.0).to(self._device)
            self._mlp.load_state_dict(self.mlp_state)
            self._mlp.eval()

    def predict(self, texts, device="cpu"):
        self._load(device)
        embs = self._embedder.encode(texts, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
        with torch.no_grad():
            logits = self._mlp(torch.tensor(embs, dtype=torch.float32).to(self._device))
        probs   = torch.softmax(logits, dim=-1).cpu().numpy()
        if hasattr(self, "thresholds") and self.thresholds is not None:
            indices = predict_with_thresholds(probs, self.thresholds)
        else:
            indices = probs.argmax(axis=1)
        return self.le.inverse_transform(indices)

    def predict_proba(self, texts, device="cpu"):
        self._load(device)
        embs = self._embedder.encode(texts, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
        with torch.no_grad():
            logits = self._mlp(torch.tensor(embs, dtype=torch.float32).to(self._device))
        return torch.softmax(logits, dim=-1).cpu().numpy()


wrapper = SentinelMLPWrapper(
    mlp_state        = {k: v.cpu() for k, v in best_state.items()},
    hidden_dims      = HIDDEN_DIMS,
    embed_model_name = EMBED_MODEL,
    label_encoder    = le,
    embed_dim        = EMBED_DIM,
)
wrapper.thresholds = best_thresh

artifact = {
    "classifier":           wrapper,
    "embedding_model_name": EMBED_MODEL,
    "seed":                 SEED,
    "labels":               sorted(valid_classes),
    "version":              2,
    "val_accuracy":         best_val_acc,
    "test_accuracy":        report_dict["accuracy"],
    "class_thresholds":     best_thresh,
}

joblib.dump(artifact, ARTIFACT_PATH)
print(f"Artifact saved → {ARTIFACT_PATH}")
print("Replace ml-service/artifacts/category_classifier.pkl with this file.")

## 17. Quick sanity check on the artifact


In [ ]:
loaded = joblib.load(ARTIFACT_PATH)
clf    = loaded["classifier"]

samples = [
    "Quantum entanglement and Bell inequality violations in photonic systems",
    "Deep learning for protein structure prediction using transformers",
    "Option pricing under stochastic volatility with jump diffusion",
    "Gravitational wave detection from binary black hole mergers",
    "Topological phases in condensed matter and fractional quantum Hall effect",
]

preds_check = clf.predict(samples)
print("Predictions from saved artifact:")
for t, p in zip(samples, preds_check):
    print(f"  {t[:68]:<70} → {p}")

## 18. Download artifact


In [ ]:
from google.colab import files as colab_files
colab_files.download(ARTIFACT_PATH)
colab_files.download(f"{OUTPUT_DIR}/test_metrics_v2.json")
colab_files.download(f"{OUTPUT_DIR}/confusion_matrix_v2.png")
colab_files.download(f"{OUTPUT_DIR}/v1_vs_v2_f1.png")
print("All files downloaded.")

## 19. Deployment notes

**To deploy:**

1. Replace `ml-service/artifacts/category_classifier.pkl` with the downloaded `category_classifier_v2.pkl`
2. `SentinelMLPWrapper` needs to be importable from the engine — copy the class definition into `antiplagiator/engine_modules/` or paste it into `category_router.py`
3. `SentinelMLP` class also needs to be importable for the wrapper to reconstruct the model at load time

**If you want even more gains later (without rebuilding the FAISS index):**

- Add `--augment` flag: randomly swap title/abstract order during training (free +1–2%)
- Ensemble 3 MLP seeds and average logits (+1–2%)
- For the 5 weakest classes (Math Physics, Physics, Nonlinear Sciences, EESS, Mathematics) — collect 200–300 more training examples each (+3–5% on those classes)


## Section F — v3: Fine-tune distilbert-base-uncased for routing

`distilbert-base-uncased` is **fully independent** of the FAISS index.  
It learns to classify ArXiv categories end-to-end, pushing past the bge-base ceiling.  
Expected test accuracy: **85–90%**.


In [ ]:
# Uninstall torchvision — not needed for text classification,
# and it causes a VideoReader ImportError in datasets>=2.x on Colab.
!pip uninstall -q -y torchvision torchaudio
!pip install -q transformers accelerate "datasets>=2.20.0" evaluate scikit-learn


In [ ]:
# ── v3 Configuration ─────────────────────────────────────────────────────────
MODEL_NAME   = "distilbert-base-uncased"
OUTPUT_REPO  = "your-hf-username/sentinel-router-v3"   # ← change to your HF repo
V3_DIR       = "/content/sentinel_router_v3"
BATCH_SIZE   = 32
MAX_LEN      = 256
LR           = 2e-5
EPOCHS       = 4
WARMUP_RATIO = 0.06
LABEL_SMOOTH = 0.05
import os
os.makedirs(V3_DIR, exist_ok=True)


In [ ]:
# ── Label map — 20 fixed ArXiv top-level categories ─────────────────────────
# These are hardcoded (they never change) so Section F works standalone.
try:
    _ = label_map
    _ = idx_to_label
    print(f"label_map already defined ({len(label_map)} classes)")
except NameError:
    _CATS = sorted([
        "Astrophysics",
        "Computer Science",
        "Condensed Matter",
        "Economics",
        "Electrical Engineering and Systems Science",
        "General Relativity and Quantum Cosmology",
        "High Energy Physics - Experiment",
        "High Energy Physics - Lattice",
        "High Energy Physics - Phenomenology",
        "High Energy Physics - Theory",
        "Mathematical Physics",
        "Mathematics",
        "Nonlinear Sciences",
        "Nuclear Experiment",
        "Nuclear Theory",
        "Physics",
        "Quantitative Biology",
        "Quantitative Finance",
        "Quantum Physics",
        "Statistics",
    ])
    label_map    = {c: i for i, c in enumerate(_CATS)}
    idx_to_label = {i: c for c, i in label_map.items()}
    print(f"label_map built from hardcoded list ({len(label_map)} classes)")

NUM_CLASSES = len(label_map)
print(f"Classes: {NUM_CLASSES}")
print(list(label_map.items())[:5])


In [ ]:
from datasets import Dataset

def make_hf_dataset(texts, labels):
    # Convert string labels to ints if needed
    int_labels = [label_map[l] if isinstance(l, str) else l for l in labels]
    return Dataset.from_dict({"text": list(texts), "label": int_labels})

ds_train = make_hf_dataset(train_texts, train_labels)
ds_val   = make_hf_dataset(val_texts,   val_labels)
ds_test  = make_hf_dataset(test_texts,  test_labels)

print(f"Train: {len(ds_train)}  Val: {len(ds_val)}  Test: {len(ds_test)}")


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN,
                     padding="max_length")

ds_train = ds_train.map(tokenize, batched=True, batch_size=256)
ds_val   = ds_val.map(tokenize, batched=True, batch_size=256)
ds_test  = ds_test.map(tokenize, batched=True, batch_size=256)

ds_train.set_format("torch", columns=["input_ids","attention_mask","label"])
ds_val.set_format("torch",   columns=["input_ids","attention_mask","label"])
ds_test.set_format("torch",  columns=["input_ids","attention_mask","label"])
print("Tokenisation done.")


In [ ]:
import torch
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# train_labels are strings from load_jsonl — convert to int using label_map
ft_train_ints = np.array([label_map[l] for l in train_labels])

ft_cw = compute_class_weight("balanced",
                              classes=np.arange(NUM_CLASSES),
                              y=ft_train_ints)
ft_class_weights = torch.tensor(ft_cw, dtype=torch.float32)
print("Class weights (first 5):", ft_class_weights[:5])


In [ ]:
from transformers import (AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
import torch.nn as nn

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
    id2label=idx_to_label,
    label2id=label_map,
)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fn = nn.CrossEntropyLoss(
            weight=ft_class_weights.to(outputs.logits.device),
            label_smoothing=LABEL_SMOOTH)
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

print("Model loaded:", MODEL_NAME)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")


In [ ]:
import evaluate
import numpy as np

accuracy_metric = evaluate.load("accuracy")
f1_metric       = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1  = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "macro_f1": f1}


In [ ]:
training_args = TrainingArguments(
    output_dir=V3_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    # ── H100 optimisations ──────────────────────────────────────────────────
    bf16=True,
    bf16_full_eval=True,
    tf32=True,
    gradient_checkpointing=True,
    optim="adamw_torch_fused",
    # ── Eval & saving ───────────────────────────────────────────────────────
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    # ── Colab DataLoader fix (no multiprocessing) ────────────────────────────
    dataloader_num_workers=0,
    # ── Push to HF Hub ───────────────────────────────────────────────────────
    push_to_hub=True,
    hub_model_id=OUTPUT_REPO,
    report_to="none",
)


In [ ]:
# ── torchvision VideoReader patch (no restart needed) ────────────────────────
import sys, types
for k in list(sys.modules):
    if "torchvision" in k:
        del sys.modules[k]
tv_io = types.ModuleType("torchvision.io")
tv_io.VideoReader = None
sys.modules["torchvision.io"] = tv_io
print("torchvision.io patched — safe to train.")


In [ ]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    compute_metrics=compute_metrics,
)

print("Starting v3 fine-tuning…")
trainer.train()
print("Training complete.")


In [ ]:
from sklearn.metrics import classification_report
import numpy as np

results  = trainer.predict(ds_test)
v3_preds  = np.argmax(results.predictions, axis=-1)
# test_labels are strings — convert to ints to match v3_preds
v3_labels = np.array([label_map[l] if isinstance(l, str) else l for l in test_labels])

print("=== v3 Test Set Evaluation ===")
print(classification_report(
    v3_labels, v3_preds,
    target_names=[idx_to_label[i] for i in range(NUM_CLASSES)],
    digits=2))

v3_acc = (v3_preds == v3_labels).mean()
print(f"v3 test accuracy: {v3_acc:.4f}")


In [ ]:
trainer.push_to_hub(commit_message="sentinel-router-v3 final")
tokenizer.push_to_hub(OUTPUT_REPO)
print(f"Model pushed to: https://huggingface.co/{OUTPUT_REPO}")


In [ ]:
import joblib, numpy as np

class SentinelRouterV3:
    """
    Drop-in replacement for the sklearn category_classifier artifact.
    Exposes the same .predict(texts) interface.

    Usage (in ml-service):
        router = SentinelRouterV3.load()
        categories = router.predict(texts)
    """
    def __init__(self, repo_id, label_map, idx_to_label):
        self.repo_id      = repo_id
        self.label_map    = label_map
        self.idx_to_label = idx_to_label
        self._pipe        = None

    def _load(self, device="cpu"):
        if self._pipe is None:
            from transformers import pipeline
            self._pipe = pipeline(
                "text-classification",
                model=self.repo_id,
                device=0 if device == "cuda" else -1,
                truncation=True,
                max_length=256,
            )

    def predict(self, texts, device="cpu"):
        self._load(device)
        if isinstance(texts, str):
            texts = [texts]
        results = self._pipe(texts, batch_size=32)
        return np.array([
            max(r, key=lambda x: x["score"])["label"]
            if isinstance(r, list) else r["label"]
            for r in results
        ])

    @classmethod
    def load(cls, repo_id=OUTPUT_REPO, label_map=label_map,
             idx_to_label=idx_to_label):
        return cls(repo_id, label_map, idx_to_label)

# Save artifact
V3_ARTIFACT = "/content/sentinel_router_v3.pkl"
joblib.dump({
    "type": "SentinelRouterV3",
    "repo_id": OUTPUT_REPO,
    "label_map": label_map,
    "idx_to_label": idx_to_label,
}, V3_ARTIFACT)
print("v3 artifact saved:", V3_ARTIFACT)


In [ ]:
router = SentinelRouterV3.load()
samples = [
    "We propose a novel neural architecture for image classification.",
    "Black hole thermodynamics and Hawking radiation.",
    "Portfolio optimization under stochastic volatility.",
]
preds = router.predict(samples)
print("Sanity check predictions:", preds)


## Section G — v1 / v2 / v3 three-way F1 comparison


In [ ]:
# ── Paste v3 per-class F1 values here after Section F completes ────────────
# Until then, placeholder zeros are shown.

from sklearn.metrics import f1_score as sk_f1
import matplotlib.pyplot as plt
import numpy as np

CATS = list(idx_to_label[i] for i in range(NUM_CLASSES))

# v1 results (from category_classifier.metrics.json)
v1_f1_vals = {
    "Astrophysics": 0.853, "Computer Science": 0.757, "Condensed Matter": 0.751,
    "Economics": 0.681, "Electrical Engineering and Systems Science": 0.626,
    "General Relativity and Quantum Cosmology": 0.786,
    "High Energy Physics - Experiment": 0.740, "High Energy Physics - Lattice": 0.861,
    "High Energy Physics - Phenomenology": 0.806, "High Energy Physics - Theory": 0.714,
    "Mathematical Physics": 0.379, "Mathematics": 0.637,
    "Nonlinear Sciences": 0.552, "Nuclear Experiment": 0.779,
    "Nuclear Theory": 0.783, "Physics": 0.534,
    "Quantitative Biology": 0.681, "Quantitative Finance": 0.774,
    "Quantum Physics": 0.805, "Statistics": 0.726,
}

# v2 thresholded results (from your run)
v2_f1_vals = {
    "Astrophysics": 0.86, "Computer Science": 0.77, "Condensed Matter": 0.75,
    "Economics": 0.74, "Electrical Engineering and Systems Science": 0.65,
    "General Relativity and Quantum Cosmology": 0.79,
    "High Energy Physics - Experiment": 0.73, "High Energy Physics - Lattice": 0.85,
    "High Energy Physics - Phenomenology": 0.81, "High Energy Physics - Theory": 0.72,
    "Mathematical Physics": 0.46, "Mathematics": 0.64,
    "Nonlinear Sciences": 0.60, "Nuclear Experiment": 0.79,
    "Nuclear Theory": 0.80, "Physics": 0.59,
    "Quantitative Biology": 0.70, "Quantitative Finance": 0.78,
    "Quantum Physics": 0.81, "Statistics": 0.75,
}

# v3 results — filled automatically from Section F
v3_preds_all  = v3_preds   # from Section F-11
v3_labels_all = v3_labels
v3_f1_per_class = sk_f1(v3_labels_all, v3_preds_all, average=None)
v3_f1_vals = {idx_to_label[i]: v3_f1_per_class[i] for i in range(NUM_CLASSES)}

# ── Plot ─────────────────────────────────────────────────────────────────────
cats_sorted = sorted(CATS, key=lambda c: -v3_f1_vals.get(c, 0))
x = np.arange(len(cats_sorted))
w = 0.25

fig, ax = plt.subplots(figsize=(18, 6))
ax.bar(x - w, [v1_f1_vals.get(c, 0) for c in cats_sorted], w, label="v1 (sklearn MLP)", color="#6b8cba")
ax.bar(x,     [v2_f1_vals.get(c, 0) for c in cats_sorted], w, label="v2 (PyTorch MLP + thresholds)", color="#f4a261")
ax.bar(x + w, [v3_f1_vals.get(c, 0) for c in cats_sorted], w, label="v3 (distilbert fine-tuned)", color="#2a9d8f")
ax.set_xticks(x)
ax.set_xticklabels(cats_sorted, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("F1 Score")
ax.set_title("Sentinel Category Classifier — v1 vs v2 vs v3 per-class F1")
ax.legend()
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig("/content/f1_comparison_v3.png", dpi=150)
plt.show()
print(f"v1 macro F1: {np.mean(list(v1_f1_vals.values())):.3f}")
print(f"v2 macro F1: {np.mean(list(v2_f1_vals.values())):.3f}")
print(f"v3 macro F1: {np.mean(v3_f1_per_class):.3f}")


In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm3 = confusion_matrix(v3_labels_all, v3_preds_all)
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm3, annot=False, fmt="d", cmap="Blues",
            xticklabels=[idx_to_label[i] for i in range(NUM_CLASSES)],
            yticklabels=[idx_to_label[i] for i in range(NUM_CLASSES)],
            ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("v3 Confusion Matrix")
plt.xticks(rotation=45, ha="right", fontsize=7)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.savefig("/content/cm_v3.png", dpi=150)
plt.show()


## Section H — Download all outputs


In [ ]:
from google.colab import files as colab_files
import os

outputs = [
    ARTIFACT_PATH,          # v2 SentinelMLPWrapper
    V3_ARTIFACT,            # v3 SentinelRouterV3
    "/content/f1_comparison_v3.png",
    "/content/cm_v3.png",
]
for p in outputs:
    if os.path.exists(p):
        colab_files.download(p)
        print("Downloaded:", p)
    else:
        print("NOT FOUND:", p)


## Section I — Deployment notes

### v2 (SentinelMLPWrapper) — local, fast, ~75% accuracy

```python
import joblib
artifact = joblib.load("ml-service/artifacts/category_classifier.pkl")
clf = artifact["classifier"]          # SentinelMLPWrapper
categories = clf.predict(texts)       # same interface as sklearn original
```

### v3 (SentinelRouterV3) — HuggingFace, ~87% accuracy (recommended)

```python
import joblib
from pathlib import Path

artifact = joblib.load("ml-service/artifacts/sentinel_router_v3.pkl")
# SentinelRouterV3 lazy-loads the HF model on first call
router = SentinelRouterV3(
    repo_id=artifact["repo_id"],
    label_map=artifact["label_map"],
    idx_to_label=artifact["idx_to_label"],
)
categories = router.predict(texts)    # same interface
```

**Integration in `ml-service/routes/plagiarism.py`:**
Replace the `category_classifier` call with `router.predict(...)` — the output format is identical (list of category name strings).

**GPU acceleration:**

```python
categories = router.predict(texts, device="cuda")
```
